In [3]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv(
    "../data/transactions.csv",
    low_memory=False
)


print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 1,000,000
Columns: 17


In [4]:
# ==============================
# 1. Missing Values
# ==============================

missing = df.isnull().sum()

print("Missing values:")
print(missing[missing > 0])

Missing values:
user_id        98
amount         99
merchant_id    89
dtype: int64


In [5]:
# ==============================
# 2. Duplicate Transaction IDs
# ==============================

duplicate_ids = df["transaction_id"].duplicated().sum()

print("Duplicate transaction IDs:", duplicate_ids)

Duplicate transaction IDs: 118


In [6]:
# ==============================
# 3. Data Types
# ==============================

print(df.dtypes)

transaction_id                str
user_id                       str
timestamp                     str
amount                    float64
merchant_id                   str
merchant_category             str
location                      str
device_id                     str
ip_address                    str
payment_method                str
account_age                 int64
transaction_count_24h     float64
avg_transaction_amount    float64
distance_from_home        float64
is_new_device                 str
is_new_location               str
is_fraud                      str
dtype: object


In [7]:
# ==============================
# 4. Unique Values in Boolean Columns
# ==============================

for col in ["is_new_device", "is_new_location", "is_fraud"]:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False))


===== is_new_device =====
is_new_device
False      879721
True       120242
2              12
TRUE            6
unknown         5
yes             5
no              5
FALSE           4
Name: count, dtype: int64

===== is_new_location =====
is_new_location
False      879734
True       120236
unknown         6
FALSE           5
TRUE            5
2               5
yes             5
no              4
Name: count, dtype: int64

===== is_fraud =====
is_fraud
False      699977
True       299990
yes             9
2               8
TRUE            7
FALSE           4
unknown         3
no              2
Name: count, dtype: int64


In [8]:
# ==============================
# 5. Numeric Data Validation
# ==============================

numeric_cols = [
    "amount",
    "account_age",
    "transaction_count_24h",
    "avg_transaction_amount",
    "distance_from_home"
]

for col in numeric_cols:
    print(f"\n===== {col} =====")
    print("Missing:", df[col].isna().sum())
    print("Negative:", (df[col] < 0).sum())
    print("Minimum:", df[col].min())
    print("Maximum:", df[col].max())


===== amount =====
Missing: 99
Negative: 99
Minimum: -172440.08
Maximum: 250000.0

===== account_age =====
Missing: 0
Negative: 0
Minimum: 30
Maximum: 2999

===== transaction_count_24h =====
Missing: 0
Negative: 0
Minimum: 0.0
Maximum: 56.0

===== avg_transaction_amount =====
Missing: 0
Negative: 0
Minimum: 109.76
Maximum: 21502.43

===== distance_from_home =====
Missing: 0
Negative: 85
Minimum: -2047.9
Maximum: 2500.0


In [9]:
# ==============================
# 6. Transaction Amount Anomalies
# ==============================

print("Amount <= 0:", (df["amount"] <= 0).sum())

print("\nExtreme transaction amounts:")
print(
    df.nlargest(10, "amount")[
        ["transaction_id", "user_id", "amount", "is_fraud"]
    ]
)

Amount <= 0: 99

Extreme transaction amounts:
     transaction_id     user_id    amount is_fraud
832    TXN000185382  USER010458  250000.0     True
2134   TXN000484362  USER011717  250000.0     True
2196   TXN000827998  USER000092  250000.0     True
3693   TXN000771078  USER001451  250000.0     True
4019   TXN000886861  USER006649  250000.0     True
4427   TXN000076003  USER004212  250000.0     True
4808   TXN000772844  USER005983  250000.0     True
7420   TXN000310013  USER001081  250000.0     True
7552   TXN000161135  USER005630  250000.0     True
8162   TXN000237653  USER010228  250000.0     True


In [10]:
# ==============================
# 7. Distance Validation
# ==============================

print("Negative distances:", (df["distance_from_home"] < 0).sum())

print("\nLargest distances:")
print(
    df.nlargest(10, "distance_from_home")[
        ["transaction_id", "user_id", "distance_from_home", "is_fraud"]
    ]
)

Negative distances: 85

Largest distances:
       transaction_id     user_id  distance_from_home is_fraud
119465   TXN000959581  USER001562              2500.0     True
130147   TXN000596302  USER009441              2500.0    False
183331   TXN000168308  USER003715              2500.0     True
199423   TXN000139321  USER008151              2500.0     True
264601   TXN000375019  USER008882              2500.0     True
274689   TXN000253584  USER008147              2500.0    False
277046   TXN000099531  USER002797              2500.0     True
360206   TXN000487239  USER007395              2500.0     True
775294   TXN000848969  USER000468              2500.0     True
823049   TXN000562466  USER006420              2500.0     True


In [11]:
# ==============================
# 8. Categorical Validation
# ==============================

categorical_cols = [
    "merchant_category",
    "location",
    "payment_method"
]

for col in categorical_cols:
    print(f"\n===== {col} =====")
    print("Unique:", df[col].nunique(dropna=False))
    print(df[col].value_counts(dropna=False).head(20))


===== merchant_category =====
Unique: 11
merchant_category
entertainment       125910
travel              121748
electronics         119357
healthcare           93946
utilities            92217
restaurant           91028
clothing             90959
grocery              90719
retail               87790
fuel                 86221
UNKNOWN_CATEGORY       105
Name: count, dtype: int64

===== location =====
Unique: 11
location
Thiruvananthapuram    103216
Hyderabad             102338
Chennai               101930
Kozhikode             101473
Pune                  100588
Mumbai                 99216
Kolkata                98784
Kochi                  97534
Bengaluru              97491
Delhi                  97321
UNKNOWN_LOCATION         109
Name: count, dtype: int64

===== payment_method =====
Unique: 4
payment_method
upi            499939
card           300052
net_banking    150130
wallet          49879
Name: count, dtype: int64


In [12]:
# ==============================
# 9. Timestamp Validation
# ==============================

timestamp_test = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("Invalid timestamps:", timestamp_test.isna().sum())

print("\nTimestamp range:")
print("Minimum:", timestamp_test.min())
print("Maximum:", timestamp_test.max())

Invalid timestamps: 98

Timestamp range:
Minimum: 2026-01-01 00:00:02
Maximum: 2026-09-30 23:59:53


In [13]:
# ==========================================
# 10. Boolean Normalization
# ==========================================

def normalize_boolean(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, bool):
        return int(value)

    value = str(value).strip().lower()

    if value in ["true", "yes", "1"]:
        return 1

    if value in ["false", "no", "0"]:
        return 0

    return np.nan


boolean_cols = [
    "is_new_device",
    "is_new_location",
    "is_fraud"
]

for col in boolean_cols:
    df[col + "_clean"] = df[col].apply(normalize_boolean)

In [14]:
# Check normalized values

for col in boolean_cols:
    print(f"\n===== {col}_clean =====")
    print(df[col + "_clean"].value_counts(dropna=False))


===== is_new_device_clean =====
is_new_device_clean
0.0    879730
1.0    120253
NaN        17
Name: count, dtype: int64

===== is_new_location_clean =====
is_new_location_clean
0.0    879743
1.0    120246
NaN        11
Name: count, dtype: int64

===== is_fraud_clean =====
is_fraud_clean
0.0    699983
1.0    300006
NaN        11
Name: count, dtype: int64


In [15]:
# ==========================================
# 11. Data Quality Flags
# ==========================================

df["flag_missing_amount"] = df["amount"].isna()

df["flag_negative_amount"] = df["amount"] < 0

df["flag_missing_user"] = df["user_id"].isna()

df["flag_missing_merchant"] = df["merchant_id"].isna()

df["flag_negative_distance"] = (
    df["distance_from_home"] < 0
)

df["flag_duplicate_transaction"] = (
    df["transaction_id"].duplicated(keep=False)
)

df["flag_invalid_timestamp"] = (
    pd.to_datetime(
        df["timestamp"],
        errors="coerce"
    ).isna()
)

df["flag_unknown_category"] = (
    df["merchant_category"] == "UNKNOWN_CATEGORY"
)

In [16]:
quality_flags = [
    "flag_missing_amount",
    "flag_negative_amount",
    "flag_missing_user",
    "flag_missing_merchant",
    "flag_negative_distance",
    "flag_duplicate_transaction",
    "flag_invalid_timestamp",
    "flag_unknown_category"
]

df["data_quality_issue"] = df[quality_flags].any(axis=1)

print(
    "Records with at least one data-quality issue:",
    df["data_quality_issue"].sum()
)

print(
    "Percentage:",
    round(df["data_quality_issue"].mean() * 100, 4),
    "%"
)

Records with at least one data-quality issue: 909
Percentage: 0.0909 %


In [17]:
# ==========================================
# 12. Data Quality Issue Summary
# ==========================================

for col in quality_flags:
    count = df[col].sum()
    percentage = count / len(df) * 100

    print(
        f"{col:<35} "
        f"{count:>5,} "
        f"({percentage:.4f}%)"
    )

flag_missing_amount                    99 (0.0099%)
flag_negative_amount                   99 (0.0099%)
flag_missing_user                      98 (0.0098%)
flag_missing_merchant                  89 (0.0089%)
flag_negative_distance                 85 (0.0085%)
flag_duplicate_transaction            236 (0.0236%)
flag_invalid_timestamp                 98 (0.0098%)
flag_unknown_category                 105 (0.0105%)


In [18]:
# ==========================================
# 13. Multiple Issues Per Record
# ==========================================

issue_count = df[quality_flags].sum(axis=1)

print("Records with 0 issues:", (issue_count == 0).sum())
print("Records with 1 issue :", (issue_count == 1).sum())
print("Records with 2 issues:", (issue_count == 2).sum())
print("Records with 3+ issues:", (issue_count >= 3).sum())

Records with 0 issues: 999091
Records with 1 issue : 909
Records with 2 issues: 0
Records with 3+ issues: 0


In [19]:
# ==========================================
# 14. Create Clean Dataset
# ==========================================

clean_df = df.copy()

print("Original rows:", len(clean_df))

Original rows: 1000000


In [20]:
# Convert invalid timestamps to NaT

clean_df["timestamp"] = pd.to_datetime(
    clean_df["timestamp"],
    errors="coerce"
)

print(
    "Invalid timestamps remaining:",
    clean_df["timestamp"].isna().sum()
)

Invalid timestamps remaining: 98


In [21]:
# ==========================================
# Boolean Cleaning
# ==========================================

clean_df["is_new_device"] = clean_df["is_new_device_clean"]
clean_df["is_new_location"] = clean_df["is_new_location_clean"]
clean_df["is_fraud"] = clean_df["is_fraud_clean"]

# Remove temporary columns
clean_df.drop(
    columns=[
        "is_new_device_clean",
        "is_new_location_clean",
        "is_fraud_clean"
    ],
    inplace=True
)

In [22]:
# Negative transaction amounts are invalid
clean_df.loc[
    clean_df["amount"] < 0,
    "amount"
] = np.nan

# Negative distance is invalid
clean_df.loc[
    clean_df["distance_from_home"] < 0,
    "distance_from_home"
] = np.nan

In [23]:
clean_df["merchant_category"] = (
    clean_df["merchant_category"]
    .replace("UNKNOWN_CATEGORY", "unknown")
)

In [24]:
print("Cleaned shape:", clean_df.shape)

print("\nData types:")
print(clean_df.dtypes)

print("\nFraud labels:")
print(clean_df["is_fraud"].value_counts(dropna=False))

print("\nNegative amounts:")
print((clean_df["amount"] < 0).sum())

print("\nNegative distances:")
print((clean_df["distance_from_home"] < 0).sum())

Cleaned shape: (1000000, 26)

Data types:
transaction_id                           str
user_id                                  str
timestamp                     datetime64[us]
amount                               float64
merchant_id                              str
merchant_category                        str
location                                 str
device_id                                str
ip_address                               str
payment_method                           str
account_age                            int64
transaction_count_24h                float64
avg_transaction_amount               float64
distance_from_home                   float64
is_new_device                        float64
is_new_location                      float64
is_fraud                             float64
flag_missing_amount                     bool
flag_negative_amount                    bool
flag_missing_user                       bool
flag_missing_merchant                   bool
flag_negative

In [25]:
# ==========================================
# 20. Final Cleaning Verification
# ==========================================

print("========== CLEANING VERIFICATION ==========")

print("\nShape:")
print(clean_df.shape)

print("\nFraud labels:")
print(clean_df["is_fraud"].value_counts(dropna=False))

print("\nNegative amounts remaining:")
print((clean_df["amount"] < 0).sum())

print("\nNegative distances remaining:")
print((clean_df["distance_from_home"] < 0).sum())

print("\nInvalid timestamps remaining:")
print(clean_df["timestamp"].isna().sum())

print("\nUnknown merchant categories:")
print(
    (clean_df["merchant_category"] == "UNKNOWN_CATEGORY").sum()
)

print("\nBoolean values:")
for col in ["is_new_device", "is_new_location", "is_fraud"]:
    print(f"\n{col}:")
    print(clean_df[col].value_counts(dropna=False))

========== CLEANING VERIFICATION ==========

Shape:
(1000000, 26)

Fraud labels:
is_fraud
0.0    699983
1.0    300006
NaN        11
Name: count, dtype: int64

Negative amounts remaining:
0

Negative distances remaining:
0

Invalid timestamps remaining:
98

Unknown merchant categories:
0

Boolean values:

is_new_device:
is_new_device
0.0    879730
1.0    120253
NaN        17
Name: count, dtype: int64

is_new_location:
is_new_location
0.0    879743
1.0    120246
NaN        11
Name: count, dtype: int64

is_fraud:
is_fraud
0.0    699983
1.0    300006
NaN        11
Name: count, dtype: int64


In [26]:
# ==========================================
# 21. Create Model-Ready Dataset
# ==========================================

model_df = clean_df[
    clean_df["timestamp"].notna() &
    clean_df["is_fraud"].notna()
].copy()

print("Clean dataset:", len(clean_df))
print("Model-ready dataset:", len(model_df))

print("\nRows excluded from modeling:")
print(len(clean_df) - len(model_df))

Clean dataset: 1000000
Model-ready dataset: 999891

Rows excluded from modeling:
109


In [27]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

In [28]:
# Save complete cleaned dataset
clean_df.to_csv(
    processed_dir / "transactions_clean.csv",
    index=False
)

# Save model-ready dataset
model_df.to_csv(
    processed_dir / "transactions_model_ready.csv",
    index=False
)

print("Files saved successfully.")

Files saved successfully.
